# KO / PEP Pairs Trading — Phase 1 Results

Loads the most recent Parquet output from `scripts/run_backtest.py` and visualizes:
- Equity curve vs buy/hold KO and buy/hold PEP
- Drawdown chart
- Spread z-score with entry/exit bands
- Metrics table

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from backend.app.config import RESULTS_DIR
from backend.app.data.fetcher import get_pair
from backend.app.backtest.metrics import format_table, summarize

In [ ]:
# Load latest equity Parquet
equity_files = sorted([p for p in RESULTS_DIR.glob('KOPEP_*.parquet') if 'trades' not in p.name and 'folds' not in p.name])
assert equity_files, 'No KO/PEP results found. Run `make backtest` first.'
latest = equity_files[-1]
print('Loading', latest.name)
equity = pd.read_parquet(latest)['equity']

trades_path = latest.with_name(latest.stem + '_trades.parquet')
trades = pd.read_parquet(trades_path) if trades_path.exists() else pd.DataFrame()
folds_path = latest.with_name(latest.stem + '_folds.parquet')
folds = pd.read_parquet(folds_path)

In [ ]:
# Equity curve vs buy/hold benchmarks
ko, pep = get_pair('KO', 'PEP', '2014-01-01', '2024-12-31')
bench = pd.DataFrame({
    'KO buy/hold': ko['Adj Close'] / ko['Adj Close'].iloc[0] * equity.iloc[0],
    'PEP buy/hold': pep['Adj Close'] / pep['Adj Close'].iloc[0] * equity.iloc[0],
})
bench = bench.loc[equity.index]

fig, ax = plt.subplots(figsize=(11, 5))
equity.plot(ax=ax, label='Pairs strategy', linewidth=2)
bench.plot(ax=ax, alpha=0.6)
ax.set_title('OOS equity curve vs buy/hold benchmarks')
ax.set_ylabel('NAV ($)')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Drawdown chart
peak = equity.cummax()
dd = equity / peak - 1.0
fig, ax = plt.subplots(figsize=(11, 3))
dd.plot(ax=ax, color='crimson')
ax.fill_between(dd.index, dd, 0, color='crimson', alpha=0.2)
ax.set_title(f'Drawdown (max = {dd.min():.1%})')
ax.set_ylabel('Drawdown')
plt.tight_layout(); plt.show()

In [ ]:
# Spread z-score with entry/exit bands. Rebuild from the latest fold's frozen β.
from backend.app.strategies.pairs_trading import PairsTradingStrategy

used = folds[~folds['skipped']].iloc[-1]
strat = PairsTradingStrategy('KO', 'PEP', zscore_window=60)
train = pd.DataFrame({'KO': ko['Adj Close'], 'PEP': pep['Adj Close']}).loc[used['train_start']:used['train_end']].dropna()
strat.fit(train)

test_df = pd.DataFrame({'KO': ko['Adj Close'], 'PEP': pep['Adj Close']}).loc[used['test_start']:used['test_end']].dropna()
signals = strat.generate_signals(test_df)
fig, ax = plt.subplots(figsize=(11, 4))
signals['z'].plot(ax=ax)
for level, color in [(-2.0, 'green'), (-0.5, 'gray'), (0.5, 'gray'), (2.0, 'red'), (3.5, 'darkred'), (-3.5, 'darkgreen')]:
    ax.axhline(level, color=color, linestyle='--', alpha=0.4)
ax.set_title(f'Spread z-score over latest test fold ({used["test_start"].date()} → {used["test_end"].date()})')
ax.set_ylabel('z')
plt.tight_layout(); plt.show()

In [ ]:
metrics = summarize(equity, trades)
print(format_table(metrics))

## Commentary
_Fill in after first run._

- Net Sharpe in the expected 0.4–0.8 band? If > 2, look for look-ahead bugs.
- Max DD < 15%? If not, the circuit breaker config may need tightening.
- Trade count and avg holding period vs half-life estimates per fold?